# Building CLI Tools

Python ships with `argparse` for building command-line interfaces with argument parsing, help text, and subcommands. For larger or more user-friendly CLIs, `click` offers a decorator-based API that's terser and easier to test.

**What's inside:** `sys.argv`, `argparse` (positional args, options, flags, subcommands), and `click` (commands, options, arguments, groups).

**Learn more:** [argparse](https://docs.python.org/3/library/argparse.html) · [click](https://click.palletsprojects.com)

## Setup

In [ ]:
%pip install click

## 1. sys.argv: the raw interface

In [ ]:
import sys

# sys.argv is a list: [script_name, arg1, arg2, ...]
# Simulate what it looks like when called as: python myscript.py hello 42
simulated_argv = ['myscript.py', 'hello', '42']
script, *args = simulated_argv
print('script:', script)
print('args:', args)

## 2. argparse: positional and optional arguments

In [ ]:
import argparse

parser = argparse.ArgumentParser(
    prog='greet',
    description='Print a greeting.',
)

# positional argument: required, consumed by position
parser.add_argument('name', help='who to greet')

# optional argument: --count with a default
parser.add_argument('--count', type=int, default=1, help='number of times')

# flag: boolean switch
parser.add_argument('--shout', action='store_true', help='uppercase the output')

# parse_args([...]) simulates command-line input
args = parser.parse_args(['Alice', '--count', '3', '--shout'])
print(args)          # Namespace(name='Alice', count=3, shout=True)

msg = f'Hello, {args.name}!'
if args.shout:
    msg = msg.upper()
for _ in range(args.count):
    print(msg)

## 3. argparse: choices, types, and mutually exclusive groups

In [ ]:
import argparse

parser = argparse.ArgumentParser(prog='resize')
parser.add_argument('width',  type=int)
parser.add_argument('height', type=int)
parser.add_argument('--format', choices=['png', 'jpg', 'webp'], default='png')

# mutually exclusive: --verbose and --quiet can't be used together
group = parser.add_mutually_exclusive_group()
group.add_argument('--verbose', action='store_true')
group.add_argument('--quiet',   action='store_true')

args = parser.parse_args(['800', '600', '--format', 'jpg', '--verbose'])
print(args)

## 4. argparse: subcommands

In [ ]:
import argparse

parser = argparse.ArgumentParser(prog='git-lite')
subparsers = parser.add_subparsers(dest='command')

# 'commit' subcommand
commit_parser = subparsers.add_parser('commit', help='record changes')
commit_parser.add_argument('-m', '--message', required=True)
commit_parser.add_argument('--amend', action='store_true')

# 'push' subcommand
push_parser = subparsers.add_parser('push', help='upload changes')
push_parser.add_argument('remote', nargs='?', default='origin')
push_parser.add_argument('branch', nargs='?', default='main')

for argv in [
    ['commit', '-m', 'initial commit'],
    ['push', 'upstream', 'dev'],
    ['push'],
]:
    args = parser.parse_args(argv)
    print(args)

## 5. click: decorator-based CLI

In [ ]:
import click

@click.command()
@click.argument('name')
@click.option('--count', default=1, show_default=True, help='Number of greetings.')
@click.option('--shout', is_flag=True, help='Uppercase output.')
def greet(name, count, shout):
    """Print a friendly greeting."""
    msg = f'Hello, {name}!'
    if shout:
        msg = msg.upper()
    for _ in range(count):
        click.echo(msg)

# standalone_mode=False lets us call the command directly in a notebook
greet(['Alice', '--count', '3', '--shout'], standalone_mode=False)

## 6. click: groups and subcommands

In [ ]:
import click

@click.group()
def cli():
    """A simple file management tool."""
    pass

@cli.command()
@click.argument('filename')
@click.option('--lines', default=10, help='Number of lines to show.')
def head(filename, lines):
    """Show the first N lines of a file."""
    click.echo(f'Showing {lines} lines of {filename}')

@cli.command()
@click.argument('src')
@click.argument('dst')
@click.option('--force', is_flag=True)
def copy(src, dst, force):
    """Copy SRC to DST."""
    click.echo(f'Copying {src} -> {dst}{" (forced)" if force else ""}')

cli(['head', 'data.txt', '--lines', '5'], standalone_mode=False)
cli(['copy', 'a.txt', 'b.txt', '--force'], standalone_mode=False)

## 7. click: prompts, confirmation, and styled output

In [ ]:
import click

# click.style adds ANSI colours (rendered in terminals, stripped in notebooks)
click.echo(click.style('Success!', fg='green', bold=True))
click.echo(click.style('Warning!', fg='yellow'))
click.echo(click.style('Error!',   fg='red',   bold=True))

# click.confirm / click.prompt for interactive input
# (skipped here; they read from stdin)
click.echo(click.style('\nIn a real CLI:', bold=True))
click.echo('  confirmed = click.confirm("Continue?")')
click.echo('  name      = click.prompt("Your name", default="World")')